# 🧪 Lab 04: The WholeStageCodegen Boundary Diagnostics

Welcome to the execution-boundary autopsy bay. In this lab, we stop treating WholeStageCodegen as a yes/no property of an entire query and locate the places where one generated region ends.

**Mission Objective:** inspect one query with an `Exchange` boundary and one query with a Python evaluation boundary. We will identify the codegen stages on either side and read the physical plan as a map of execution jurisdictions.

**Deterministic Guardrail:** the examples are intentionally small and use native expressions wherever possible. The goal is plan evidence, not a performance benchmark. A boundary is a clue; it is not automatically a problem.


### Step 1: Define the diagnostic session
Adaptive Query Execution is disabled so the physical plan exposes the stage markers directly. The local session still exercises Spark's JVM planning and execution path.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import LongType

spark = (SparkSession.builder
    .master("local[2]")
    .appName("lab-04-find-the-boundaries")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "false")
    .config("spark.sql.shuffle.partitions", "2")
    .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)
print("WholeStageCodegen enabled:", spark.conf.get("spark.sql.codegen.wholeStage"))


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/24 06:22:45 WARN Utils: Your hostname, T14-PF4WM3XL, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/08/24 06:22:45 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/08/24 06:22:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version: 4.2.0
WholeStageCodegen enabled: true


### Step 2: Put an Exchange between two generated regions
The aggregation requires data to be redistributed by `bucket`. The `Exchange` is the shuffle boundary: codegen can operate below it and above it, but cannot fuse through the redistribution itself.


In [2]:
exchange_query = (spark.range(0, 1_000_000)
    .select((F.col("id") % 16).alias("bucket"), F.col("id").alias("value"))
    .groupBy("bucket")
    .agg(F.sum("value").alias("total")))

exchange_query.collect()
print("=== Exchange boundary plan ===")
exchange_query.explain("formatted")


=== Exchange boundary plan ===
== Physical Plan ==
* HashAggregate (5)
+- Exchange (4)
   +- * HashAggregate (3)
      +- * Project (2)
         +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#0L]
Arguments: Range (0, 1000000, step=1, splits=Some(2))

(2) Project [codegen id : 1]
Output [2]: [(id#0L % 16) AS bucket#1L, id#0L AS value#2L]
Input [1]: [id#0L]

(3) HashAggregate [codegen id : 1]
Input [2]: [bucket#1L, value#2L]
Keys [1]: [bucket#1L]
Functions [1]: [partial_sum(value#2L)]
Aggregate Attributes [1]: [sum#7L]
Results [2]: [bucket#1L, sum#8L]

(4) Exchange
Input [2]: [bucket#1L, sum#8L]
Arguments: hashpartitioning(bucket#1L, 2), ENSURE_REQUIREMENTS, [plan_id=17]

(5) HashAggregate [codegen id : 2]
Input [2]: [bucket#1L, sum#8L]
Keys [1]: [bucket#1L]
Functions [1]: [sum(value#2L)]
Aggregate Attributes [1]: [sum(value#2L)#6L]
Results [2]: [bucket#1L, sum(value#2L)#6L AS total#3L]




### Step 3: Read the Exchange evidence
Find the `Exchange` in the formatted plan. The operators below it should share one codegen ID, while the aggregate above it should belong to another stage. The changed stage identity records the point where data moved and a new local pipeline began.


In [3]:
print("Exchange present:", "Exchange" in exchange_query._jdf.queryExecution().executedPlan().toString())
print("Codegen IDs are visible above in the formatted plan.")


Exchange present: True
Codegen IDs are visible above in the formatted plan.


### Step 4: Introduce a Python runtime boundary
A regular Python UDF cannot be spliced into the generated Java program. Spark inserts a Python evaluation operator, then codegen may resume for compatible JVM operators after Python returns the data.


In [4]:
@F.udf(returnType=LongType())
def add_one_in_python(value):
    return None if value is None else value + 1

python_query = (spark.range(0, 100_000)
    .select(add_one_in_python(F.col("id")).alias("x"))
    .select((F.col("x") * 2).alias("y"))
    .agg(F.sum("y").alias("checksum")))

python_query.collect()
print("=== Python boundary plan ===")
python_query.explain("formatted")


/home/angelalvarez/.local/lib/python3.12/site-packages/pyspark/sql/udf.py:116: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


=== Python boundary plan ===
== Physical Plan ==
* HashAggregate (6)
+- Exchange (5)
   +- * HashAggregate (4)
      +- * Project (3)
         +- ArrowEvalPython (2)
            +- * Range (1)


(1) Range [codegen id : 1]
Output [1]: [id#10L]
Arguments: Range (0, 100000, step=1, splits=Some(2))

(2) ArrowEvalPython
Input [1]: [id#10L]
Arguments: [add_one_in_python(id#10L)#11L], [pythonUDF0#17L], 101

(3) Project [codegen id : 2]
Output [1]: [(pythonUDF0#17L * 2) AS y#13L]
Input [2]: [id#10L, pythonUDF0#17L]

(4) HashAggregate [codegen id : 2]
Input [1]: [y#13L]
Keys: []
Functions [1]: [partial_sum(y#13L)]
Aggregate Attributes [1]: [sum#18L]
Results [1]: [sum#19L]

(5) Exchange
Input [1]: [sum#19L]
Arguments: SinglePartition, ENSURE_REQUIREMENTS, [plan_id=55]

(6) HashAggregate [codegen id : 3]
Input [1]: [sum#19L]
Keys: []
Functions [1]: [sum(y#13L)]
Aggregate Attributes [1]: [sum(y#13L)#16L]
Results [1]: [sum(y#13L)#16L AS checksum#14L]




### Step 5: Identify the Python boundary
Look for `BatchEvalPython` or `ArrowEvalPython`. The surrounding codegen IDs show which JVM regions remain code-generated; the Python node marks the jurisdiction change. Arrow can reduce transfer overhead, but it does not turn arbitrary Python function bodies into generated Java.


In [5]:
executed_python_plan = python_query._jdf.queryExecution().executedPlan().toString()
print("Python boundary present:", "BatchEvalPython" in executed_python_plan or "ArrowEvalPython" in executed_python_plan)
print("Python evaluation operator visible in formatted output: inspect the plan above.")


Python boundary present: True
Python evaluation operator visible in formatted output: inspect the plan above.


# 📊 Post-Lab Analysis: Where the Pipeline Breaks

This lab showed that WholeStageCodegen is composed of local execution islands rather than one promise covering the whole query. The physical plan identifies the boundary, the operator that caused it, and the place where generated execution resumes.

### 1. Exchange Starts a New Local Jurisdiction

The aggregation plan contains an `Exchange` between the pre-shuffle operators and the final aggregation. The codegen IDs change across that point because Spark must redistribute data before the next local pipeline can consume it. No generated Java loop can fuse through a shuffle boundary.

### 2. Python Speaks for Itself

The Python UDF plan contains a Python evaluation operator between JVM execution regions. The surrounding native expressions can still participate in codegen, but the Python function remains in a separate runtime. Arrow may make the crossing cheaper; it does not remove the crossing.

### 3. A Boundary Is Evidence, Not a Verdict

Neither `Exchange` nor a Python evaluation node is automatically a defect. An exchange may be required by the algorithm, and a Python UDF may be the correct engineering choice. The boundary tells us where to investigate; runtime metrics tell us whether its cost matters.

The useful question is no longer simply whether codegen is enabled. It is: **where did Spark have to stop writing the same program, and what happened there?**
